### Phase 2: Silver Layer – Data Cleaning, Enrichment and Validation

The Silver layer transforms raw Bronze data into structured and reliable datasets.

Key processing steps:

• Data quality validation checks  
• Geospatial enrichment using Geo-IP lookup  
• Velocity tracking to detect suspicious activity  
• CDC implementation using MERGE to maintain latest account state  

The goal is to ensure only validated and enriched data flows to the Gold analytics layer.

#### Load Bronze

In [0]:
transfers = spark.table("bronze_transfers_raw")
accounts = spark.table("bronze_account_metadata")
geo = spark.table("bronze_geo_ip_lookup")

#### ->Geospatial Enrichment

In [0]:
from pyspark.sql.functions import broadcast

transfers_geo = transfers.join(
    broadcast(geo),
    transfers.ip_address.startswith(geo.ip_prefix),
    "left"
)

#### ->Velocity Tracking (1-hour window)

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import unix_timestamp, col, count

# convert timestamp to seconds
transfers_geo = transfers_geo.withColumn(
    "transfer_time_sec",
    unix_timestamp(col("transfer_time"))
)

# window definition
w = Window.partitionBy("account_id") \
.orderBy(col("transfer_time_sec")) \
.rangeBetween(-3600,0)

# velocity tracking
velocity = transfers_geo.withColumn(
    "txn_last_hour",
    count("*").over(w)
)

In [0]:
velocity_dedup = velocity.dropDuplicates(["transfer_id"])

In [0]:
velocity_dedup.write.format("delta") \
.mode("overwrite") \
.saveAsTable("silver_transfers_enriched")

#### ->CDC Account Balance

In [0]:
from pyspark.sql.functions import sum

balances = velocity_dedup.groupBy("account_id").agg(
    sum("amount").alias("total_transferred")
)

balances.write.format("delta") \
.mode("overwrite") \
.saveAsTable("silver_account_balance_cdc")